# Major Sovereign Bond Yield Overview

Import major sovereign yield curves across large developed and systemically important markets and visualize how the short end and the 10-year sector have behaved over the last 10 years.

## What This Notebook Uses
- **Short end**: 3-month interbank rate from FRED/OECD as the policy-sensitive front-end anchor.
- **Long end**: 10-year government bond yield from FRED/OECD.
- **Countries**: United States, Germany, United Kingdom, Japan, France, Canada, Australia, and India.

## Important Note
Exact cross-country **2-year** sovereign yield series are not exposed consistently enough through FRED/OECD to build a stable multi-country notebook. To keep the comparison robust, this notebook uses the **3-month rate as the short-end proxy** and the **10-year government yield as the long-end anchor**. That still captures the key behaviour of each sovereign curve: front-end policy pressure, long-end term pricing, and curve steepness.

## Why This Matters
Taken together, these series let you see where rate pressure is concentrated, which countries have the highest or lowest long-end yields, and whether curves are steepening or inverting over time.

In [ ]:
%pip install pandas plotly requests

In [1]:
import io
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from plotly.subplots import make_subplots

bond_definitions = [
    {
        "name": "United States",
        "group": "North America",
        "short_series": "IR3TIB01USM156N",
        "long_series": "IRLTLT01USM156N",
        "note": "3M interbank rate and 10Y government bond yield from FRED/OECD.",
    },
    {
        "name": "Canada",
        "group": "North America",
        "short_series": "IR3TIB01CAM156N",
        "long_series": "IRLTLT01CAM156N",
        "note": "3M interbank rate and 10Y government bond yield from FRED/OECD.",
    },
    {
        "name": "Germany",
        "group": "Europe",
        "short_series": "IR3TIB01DEM156N",
        "long_series": "IRLTLT01DEM156N",
        "note": "3M interbank rate and 10Y government bond yield from FRED/OECD.",
    },
    {
        "name": "France",
        "group": "Europe",
        "short_series": "IR3TIB01FRM156N",
        "long_series": "IRLTLT01FRM156N",
        "note": "3M interbank rate and 10Y government bond yield from FRED/OECD.",
    },
    {
        "name": "United Kingdom",
        "group": "Europe",
        "short_series": "IR3TIB01GBM156N",
        "long_series": "IRLTLT01GBM156N",
        "note": "3M interbank rate and 10Y government bond yield from FRED/OECD.",
    },
    {
        "name": "Japan",
        "group": "Asia Pacific",
        "short_series": "IR3TIB01JPM156N",
        "long_series": "IRLTLT01JPM156N",
        "note": "3M interbank rate and 10Y government bond yield from FRED/OECD.",
    },
    {
        "name": "Australia",
        "group": "Asia Pacific",
        "short_series": "IR3TIB01AUM156N",
        "long_series": "IRLTLT01AUM156N",
        "note": "3M interbank rate and 10Y government bond yield from FRED/OECD.",
    },
    {
        "name": "India",
        "group": "Asia Pacific",
        "short_series": "INDIR3TIB01STM",
        "long_series": "INDIRLTLT01STM",
        "note": "3M interbank rate and 10Y government bond yield from FRED/OECD.",
    },
]


def fetch_fred_series(series_id: str) -> pd.Series:
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    frame = pd.read_csv(io.StringIO(response.text))
    if frame.empty or "observation_date" not in frame.columns or series_id not in frame.columns:
        raise ValueError(f"Unexpected FRED response for {series_id}.")
    frame["observation_date"] = pd.to_datetime(frame["observation_date"])
    frame[series_id] = pd.to_numeric(frame[series_id], errors="coerce")
    return frame.set_index("observation_date")[series_id]


series_frames = {}
available_definitions = []
unavailable_names = []

for definition in bond_definitions:
    try:
        short_series = fetch_fred_series(definition["short_series"])
        long_series = fetch_fred_series(definition["long_series"])
    except Exception:
        unavailable_names.append(definition["name"])
        continue

    series_frames[f'{definition["name"]} 3M'] = short_series
    series_frames[f'{definition["name"]} 10Y'] = long_series
    available_definitions.append(definition)

if not series_frames:
    raise ValueError("No sovereign yield series could be loaded from FRED.")

rates = pd.DataFrame(series_frames).sort_index()
rates = rates.loc[rates.index >= (rates.index.max() - pd.DateOffset(years=10))]
rates = rates.dropna(axis=1, how="all").ffill()

available_names = [definition["name"] for definition in available_definitions]
short_columns = [f"{name} 3M" for name in available_names if f"{name} 3M" in rates.columns]
long_columns = [f"{name} 10Y" for name in available_names if f"{name} 10Y" in rates.columns]
short_yields = rates[short_columns].copy()
long_yields = rates[long_columns].copy()
short_yields.columns = [column.removesuffix(" 3M") for column in short_columns]
long_yields.columns = [column.removesuffix(" 10Y") for column in long_columns]
curve_spread = long_yields.subtract(short_yields, fill_value=pd.NA)
long_price_proxy = np.exp(-(long_yields / 100) * 10)
long_price_proxy = long_price_proxy.apply(
    lambda column: column / column.dropna().iloc[0] * 100 if not column.dropna().empty else column
)

latest_snapshot = pd.DataFrame(
    {
        "3M Yield": short_yields.iloc[-1],
        "10Y Yield": long_yields.iloc[-1],
        "10Y-3M Spread": curve_spread.iloc[-1],
        "1Y Change 3M (bps)": (short_yields.iloc[-1] - short_yields.shift(12).iloc[-1]) * 100,
        "1Y Change 10Y (bps)": (long_yields.iloc[-1] - long_yields.shift(12).iloc[-1]) * 100,
        "10Y Price Proxy 1Y Return (%)": (long_price_proxy.iloc[-1] / long_price_proxy.shift(12).iloc[-1] - 1) * 100,
    }
).sort_values("10Y Yield", ascending=False)

steepest_curves = latest_snapshot.sort_values("10Y-3M Spread", ascending=False).head(3)
flattest_curves = latest_snapshot.sort_values("10Y-3M Spread", ascending=True).head(3)
highest_long_end = latest_snapshot.sort_values("10Y Yield", ascending=False).head(3)
lowest_long_end = latest_snapshot.sort_values("10Y Yield", ascending=True).head(3)

narrative_lines = [
    f"Latest data point: {rates.index.max().date()}.",
    f"Highest 10Y yields: {', '.join(f'{name} ({value:.2f}%)' for name, value in highest_long_end['10Y Yield'].items())}.",
    f"Lowest 10Y yields: {', '.join(f'{name} ({value:.2f}%)' for name, value in lowest_long_end['10Y Yield'].items())}.",
    f"Steepest curves: {', '.join(f'{name} ({value:.2f} pp)' for name, value in steepest_curves['10Y-3M Spread'].items())}.",
    f"Flattest curves: {', '.join(f'{name} ({value:.2f} pp)' for name, value in flattest_curves['10Y-3M Spread'].items())}.",
]

summary_payload = {
    "data_source": "FRED/OECD",
    "requested_window": "10y",
    "frequency": "monthly",
    "short_end_note": "3M interbank rate used as the short-end proxy because a stable exact 2Y sovereign series is not consistently available cross-country in FRED/OECD.",
    "countries_requested": [definition["name"] for definition in bond_definitions],
    "countries_available": available_names,
    "countries_unavailable": unavailable_names,
    "date_range": {
        "start": str(rates.index.min().date()),
        "end": str(rates.index.max().date()),
    },
    "leaders": {
        "highest_10y_yields": highest_long_end["10Y Yield"].round(3).to_dict(),
        "lowest_10y_yields": lowest_long_end["10Y Yield"].round(3).to_dict(),
        "steepest_curves": steepest_curves["10Y-3M Spread"].round(3).to_dict(),
        "flattest_curves": flattest_curves["10Y-3M Spread"].round(3).to_dict(),
    },
    "latest_snapshot": latest_snapshot.round(3).reset_index().rename(columns={"index": "country"}).to_dict(orient="records"),
    "narrative": narrative_lines,
}

print("LLM_SUMMARY")
for line in narrative_lines:
    print(f"- {line}")

latest_snapshot.round(2)

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=(
        "Short-End Rates (3M Proxy)",
        "10-Year Government Bond Yields",
        "Curve Spread (10Y minus 3M)",
        "10-Year Zero-Coupon Price Proxy (Normalized)",
    ),
)

for name in available_names:
    fig.add_trace(
        go.Scatter(x=short_yields.index, y=short_yields[name], mode="lines", name=f"{name} 3M", legendgroup=name),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=long_yields.index, y=long_yields[name], mode="lines", name=f"{name} 10Y", legendgroup=name, showlegend=False),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=curve_spread.index, y=curve_spread[name], mode="lines", name=f"{name} Spread", legendgroup=name, showlegend=False),
        row=3,
        col=1,
    )
    fig.add_trace(
        go.Scatter(x=long_price_proxy.index, y=long_price_proxy[name], mode="lines", name=f"{name} 10Y Price Proxy", legendgroup=name, showlegend=False),
        row=4,
        col=1,
    )

fig.update_yaxes(title_text="Percent", row=1, col=1)
fig.update_yaxes(title_text="Percent", row=2, col=1)
fig.update_yaxes(title_text="Pct. Points", row=3, col=1)
fig.update_yaxes(title_text="Index = 100", row=4, col=1)
fig.update_layout(
    height=1200,
    width=1200,
    hovermode="x unified",
    title="Major Sovereign Yield Behaviour Over the Last 10 Years",
    legend_title="Country / Segment",
)
fig.show()

LLM_SUMMARY
- Latest data point: 2026-03-01.
- Highest 10Y yields: India (6.78%), Australia (4.90%), United Kingdom (4.70%).
- Lowest 10Y yields: Japan (2.35%), Germany (2.91%), France (3.40%).
- Steepest curves: India (1.46 pp), France (1.29 pp), Canada (1.21 pp).
- Flattest curves: United States (0.56 pp), Australia (0.71 pp), Germany (0.80 pp).


In [3]:
yield_curve_snapshot = pd.DataFrame(
    {
        "3M": short_yields.iloc[-1].reindex(available_names),
        "10Y": long_yields.iloc[-1].reindex(available_names),
    }
)

yield_curve_snapshot = yield_curve_snapshot.dropna(how="any")
latest_curve_date = max(short_yields.index.max(), long_yields.index.max()).date()

curve_rows = int(np.ceil(len(yield_curve_snapshot) / 2))
yield_curve_fig = make_subplots(
    rows=curve_rows,
    cols=2,
    subplot_titles=[
        f"{country} ({yield_curve_snapshot.loc[country, '10Y'] - yield_curve_snapshot.loc[country, '3M']:.2f} pp slope)"
        for country in yield_curve_snapshot.index
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.12,
)

maturity_labels = ["3M", "10Y"]

for position, country in enumerate(yield_curve_snapshot.index):
    row = position // 2 + 1
    col = position % 2 + 1
    curve_values = yield_curve_snapshot.loc[country, maturity_labels]
    yield_curve_fig.add_trace(
        go.Scatter(
            x=maturity_labels,
            y=curve_values,
            mode="lines+markers+text",
            name=country,
            text=[f"{value:.2f}%" for value in curve_values],
            textposition="top center",
            line=dict(width=3),
            marker=dict(size=9),
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    yield_curve_fig.update_yaxes(title_text="Percent", row=row, col=col)

for empty_position in range(len(yield_curve_snapshot.index), curve_rows * 2):
    row = empty_position // 2 + 1
    col = empty_position % 2 + 1
    yield_curve_fig.update_xaxes(visible=False, row=row, col=col)
    yield_curve_fig.update_yaxes(visible=False, row=row, col=col)

yield_curve_fig.update_layout(
    height=max(500, 320 * curve_rows),
    width=1100,
    title=f"Latest Simplified Sovereign Yield Curves by Country ({latest_curve_date})",
)

yield_curve_fig.show()